## Sequential text pre-processing

In [1]:
import numpy as np
import pandas as pd
pd.set_option('display.max_colwidth', 100)

# df = pd.read_csv('./data/wyoming.csv')
df = pd.read_csv('./data/new-england.csv')
df.shape

(2873728, 3)

In [2]:
df.loc[0,['text']]

text    Loved the food. The staff member we interacted with was friendly and able to answer pur question...
Name: 0, dtype: object

In [3]:
df.head(2)

,rating,text,type
0,5,Loved the food. The staff member we interacted with was friendly and able to answer pur question...,Hot pot restaurant
1,4,Food was good. The experience is what you're paying for. Definitely worth a try!,Hot pot restaurant


In [4]:
# Added for 'new-england.csv'
df = df[df['text'].apply(lambda x: isinstance(x, str))]

In [5]:
df.shape

(2873701, 3)

### Tokenize

The Keras `Tokenizer` seems to be the most modern all-in-one solution

In [6]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

tokenizer = Tokenizer(num_words=10000, oov_token='<OOV>')
tokenizer.fit_on_texts(df['text'])
X_seq = tokenizer.texts_to_sequences(df['text'])

In [7]:
len(tokenizer.word_index) # Maximum vocabulary size (may useful for setting LSTM model vocab_size)

231364

### Verify output

In [8]:
X_seq[0]

[232,
 2,
 5,
 2,
 32,
 2175,
 28,
 6575,
 21,
 7,
 36,
 3,
 679,
 6,
 1706,
 1,
 1724,
 108,
 207,
 2,
 145,
 1134,
 1133,
 3,
 141,
 2415,
 27,
 89,
 6,
 61,
 75,
 662,
 28,
 68,
 266,
 40,
 59]

In [9]:
[tokenizer.index_word[i] for i in X_seq[0]]

['loved',
 'the',
 'food',
 'the',
 'staff',
 'member',
 'we',
 'interacted',
 'with',
 'was',
 'friendly',
 'and',
 'able',
 'to',
 'answer',
 '<OOV>',
 'questions',
 'about',
 'how',
 'the',
 'hot',
 'pot',
 'works',
 'and',
 'make',
 'suggestions',
 'on',
 'what',
 'to',
 'order',
 'no',
 'problem',
 'we',
 'will',
 'absolutely',
 'be',
 'back']

### Pad sequences

Make observations of identical size.

Note: Padding increases data size, so saved files take up more space. However, train-test split on a variable-length list object is a hassle (Alteratively, add text pre-processing to model file without saving to disk.)

Note: There is some potential for "leakage" between training and test data if padding before splitting.

In [10]:
lengths = [len(seq) for seq in X_seq]
print(f'Max length: {np.max(lengths)}')
print(f'Average length: {np.mean(lengths)}')
print(f'Quantiles: {np.quantile(lengths, [0.01, 0.10, 0.25, 0.5, 0.75, 0.90, 0.99])}')

Max length: 1750
Average length: 20.247580732999015
Quantiles: [  1.   3.   5.  10.  23.  48. 144.]


In [11]:
max_length = 50 # Shortened from 100 for 'pacific.csv'
X = pad_sequences(X_seq, maxlen=max_length, padding='post', truncating='post')

In [12]:
del X_seq

In [13]:
X[0:2]

array([[ 232,    2,    5,    2,   32, 2175,   28, 6575,   21,    7,   36,
           3,  679,    6, 1706,    1, 1724,  108,  207,    2,  145, 1134,
        1133,    3,  141, 2415,   27,   89,    6,   61,   75,  662,   28,
          68,  266,   40,   59,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0],
       [   5,    7,   11,    2,  121,   10,   89,  325, 1114,   12,  107,
         151,    4,  112,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0]], dtype=int32)

### Train-test split

In [14]:
y = df['rating'] - 1 # shold be 0-4 indexed

In [15]:
del df

In [16]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [17]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((2298960, 50), (2298960,), (574741, 50), (574741,))

In [18]:
# np.save('./data/X_train_seq.npy', X_train)
# np.save('./data/y_train_seq.npy', y_train)

# np.save('./data/X_test_seq.npy', X_test)
# np.save('./data/y_test_seq.npy', y_test)

np.save('./data/X_train_seq_lrg.npy', X_train)
np.save('./data/y_train_seq_lrg.npy', y_train)

np.save('./data/X_test_seq_lrg.npy', X_test)
np.save('./data/y_test_seq_lrg.npy', y_test)